In [1]:
import numpy as np
import time
import os
import ase
from pyscf import gto, dft, df, lib
from pyscf.scf import hf
import scipy
from equiv_dens.utils import base as utils
%cd /home/mihail/Documents/workspace/equiv_dens/
hf.MUTE_CHKFILE = True
%load_ext autoreload
%autoreload 2

loading config file None
/home/mihail/Documents/workspace/equiv_dens


In [ ]:
# mol = gto.M(atom='O  0  0  0.1184; H  0,  0.7532, -0.4735; H 0,  -0.7532, -0.4735 ', basis='def2svp')
# mf = dft.RKS(mol)
# mf.chkfile=False
# mf.xc = 'pbe'
# mf.kernel()
# g = mf.nuc_grad_method()
# g.kernel()
data = np.load('datasets/h2o_dynamic_centered.npy', allow_pickle=True).item()
basis = 'augccpvdz'
auxbasis = 'augccpvqzjkfit'

atom_types = data['atom_types']

print(len(data['positions']))
save_path = 'datasets/h2o_dynamic_augccpvdz_df_augccpvqzjkfit.npy'
npy_path = 'datasets/h2o_dynamic_augccpvdz.npy'
if os.path.exists(save_path):
    results = list(np.load(save_path, allow_pickle=True))
else:
    results = []
print('results len', len(results))
for i in range(len(results), len(data['positions'])):
    print('calc', i)
    start = time.time()
    pos = data['positions'][i]
    atom = []
    for j in range(len(atom_types)):
        atom.append((atom_types[j], pos[j, :])) 
    mol = gto.M(atom=atom, basis=basis)
    #print(mol.pack())
    mf = dft.RKS(mol)
    mf.chkfile=False
    mf.xc = 'pbe'
    mf.kernel()
    g = mf.nuc_grad_method()
    gradients = g.grad()
    print('elapsed', time.time() - start)
    #print(mfs[i].mo_coeff)
    res = []
    res.append(mol.pack())
    calc_dict = {}
    calc_dict['mo_coeff'] = mf.mo_coeff
    calc_dict['mo_occ'] = mf.mo_occ
    calc_dict['energy'] = mf.e_tot
    calc_dict['forces'] = -gradients/ase.units.Bohr

    dm1 = mf.make_rdm1(mf.mo_coeff, mf.mo_occ)
    auxmol = df.addons.make_auxmol(mol, auxbasis)

    ints_3c2e = df.incore.aux_e2(mol, auxmol, intor='int3c2e')
    ints_2c2e = auxmol.intor('int2c2e')
    print('ints3c2e shape', ints_3c2e.shape)
    print('ints2c2e shape', ints_2c2e.shape)

    nao = mol.nao
    naux = auxmol.nao
    df_coef = scipy.linalg.solve(ints_2c2e, ints_3c2e.reshape(nao*nao, naux).T)
    df_coef = df_coef.reshape(naux, nao, nao)
    if dm1.ndim > 2:
        df_basis = []
        for j in range(dm1.shape[0]):
            df_basis.append(lib.einsum('Pij,ij->P', df_coef, dm1[j]))
        df_basis = np.stack(df_basis, axis=0)
        print(df_basis.shape)

    else:
        df_basis = lib.einsum('Pij,ij->P', df_coef, dm1)

    calc_dict['df_coeff'] = df_basis
    calc_dict['auxbasis'] = auxbasis
    res.append(calc_dict)
    results.append(res)

    if i%100 == 0:
        np.save(save_path, results, allow_pickle=True)
np.save(save_path, results, allow_pickle=True)
npy_data = utils.calc_dict_to_npy(results, convert_forces=False, compress_atoms=False)
np.save(npy_path, npy_data, allow_pickle=True)

In [ ]:
# calculating for a single molecule, and getting different energy components
data = np.load('datasets/h2o_dynamic_centered.npy', allow_pickle=True).item()
basis = 'augccpvdz'

atom_types = data['atom_types']

i = 0
print('calc', i)
start = time.time()
pos = data['positions'][i]
atom = []
for j in range(len(atom_types)):
    atom.append((atom_types[j], pos[j, :]))
mol = gto.M(atom=atom, basis=basis)
#print(mol.pack())
mf = hf.RHF(mol)
mf.max_cycle = 0
mf.init_guess = 'atom'
mf.chkfile=False
mf.kernel()
print(mf.energy_tot())
print(mf.e_tot)

In [ ]:
dm = mf.make_rdm1()
m_kin = mol.intor('int1e_kin')
m_nuc = mol.intor('int1e_nuc')

e_kin = np.einsum('ij,ji', dm, m_kin)
e_nuc = np.einsum('ij,ji', dm, m_nuc)

veff = mf.get_veff()
ecoul = veff.ecoul
exc = veff.exc

print('total energy', e_kin + e_nuc + ecoul + exc + mf.energy_nuc())
print(mf.__dir__())
print('total_energy etot', mf.e_tot)
print('total_energy', mf.energy_tot())
print('nuclear energy', mf.energy_nuc())
print('eletronic energy, coulomb energy', mf.energy_elec())
print(mf.get_veff().shape)
print(mf.mo_coeff.shape)

In [ ]:
def get_energy_components(mol, mf):
    """
    Get energy components for a single molecule.

    Args:
        mol: pyscf molecule
        mf: pyscf scf object
    Returns:
        energies: dictionary of energy components
    """
    dm = mf.make_rdm1()
    m_kin = mol.intor('int1e_kin')
    m_nuc = mol.intor('int1e_nuc')
    h1e = mf.get_hcore()
    veff = mf.get_veff()

    energies = {}
    energies['energy'] = mf.energy_tot()
    energies['energy_e_kin'] = np.einsum('ij,ji', dm, m_kin)
    energies['energy_e_nuc'] = np.einsum('ij,ji', dm, m_nuc)
    energies['energy_coul'] = veff.ecoul
    energies['energy_exc'] = veff.exc
    energies['energy_nuc'] = mf.energy_nuc()
    # print('energies', energies)
    # print('total energy', energies['energy'])
    # print('mf energy elec', mf.energy_elec())
    # print('mf energy nuc', mf.energy_nuc())
    # print('mf energy elec + nuc', mf.energy_elec() + mf.energy_nuc())
    # print('mf ecoul', energies['energy_coul'] + energies['energy_exc'])
    # print('energy h1e', energies['energy_e_kin'] + energies['energy_e_nuc'])
    # print('mf h1e', np.einsum('ij,ji', dm, h1e))
    #
    # print('total elec', energies['energy_e_kin'] + energies['energy_e_nuc'] +
    #       energies['energy_coul'] + energies['energy_exc'])
    # print('mf elec', np.einsum('ij,ji', dm, h1e) + energies['energy_coul'] + energies['energy_exc'])
    # print('summed components', energies['energy_e_kin'] + energies['energy_e_nuc'] +
    #       energies['energy_coul'] + energies['energy_exc'] + energies['energy_nuc'])

    assert np.isclose(energies['energy'], energies['energy_e_kin'] + energies['energy_e_nuc'] +
                      energies['energy_coul'] + energies['energy_exc'] + energies['energy_nuc'])
    return energies

In [8]:
set_types = ['train', 'valid', 'test']
for set_type in set_types:
    data = np.load('datasets/h2o_small_' + set_type + '_augccpvdz.npy', allow_pickle=True).item()
    basis = 'augccpvdz'
    auxbasis = 'augccpvqzjkfit'

    print(len(data['positions']))
    save_path = 'datasets/h2o_small_' + set_type + '_dft_augccpvdz_energy_comps_calc.npy'
    npy_path = 'datasets/h2o_small_' + set_type + '_dft_augccpvdz_energy_comps.npy'
    if os.path.exists(save_path):
        results = list(np.load(save_path, allow_pickle=True))
    else:
        results = []
    print('results len', len(results))
    for i in range(len(results), len(data['positions'])):
        print('calc', i)
        start = time.time()
        print('data positions shape', data['positions'].shape)
        pos = data['positions'][i]
        anums = data['atom_numbers'][i]
        print('pos shape', pos.shape)
        atom = []
        for j in range(len(anums)):
            atom.append((anums[j], pos[j, :])) 
        mol = gto.M(atom=atom, basis=basis)
        res = []
        res.append(mol.pack())
        #print(mol.pack())
        mf = dft.RKS(mol)
        mf.init_guess = 'atom'
        mf.max_cycle = 0
        mf.chkfile=False
        mf.xc = 'pbe'
        mf.kernel()
        g = mf.nuc_grad_method()
        gradients = g.grad()
        energies_SAD = get_energy_components(mol, mf)
        energies_SAD = {k + '_SAD': v for k, v in energies_SAD.items()}
        calc_dict = {}
        calc_dict.update(energies_SAD)
        calc_dict['forces_SAD'] = -gradients/ase.units.Bohr
        mol = gto.M(atom=atom, basis=basis)
        #print(mol.pack())
        mf = dft.RKS(mol)
        mf.chkfile=False
        mf.xc = 'pbe'
        mf.kernel()
        g = mf.nuc_grad_method()
        gradients = g.grad()
        energies = get_energy_components(mol, mf)

        calc_dict['forces'] = -gradients/ase.units.Bohr
        calc_dict.update(energies)

        print('calc_dict', calc_dict)
        res.append(calc_dict)
        results.append(res)

        if i%10 == 0:
            np.save(save_path, results, allow_pickle=True)
    np.save(save_path, results, allow_pickle=True)
    npy_data = utils.calc_dict_to_npy(results, convert_forces=False, compress_atoms=False)
    npy_data_compressed = utils.calc_dict_to_npy(results, convert_forces=False, compress_atoms=True)
    print('atom_number nc', npy_data['atom_numbers'][:3])
    print('atom_number c', npy_data_compressed['atom_numbers'][:3])
    print('pos nc', npy_data['positions'][:3])
    print('pos c', npy_data_compressed['positions'][:3])
    print('forces nc', npy_data['forces'][:3])
    print('forces c', npy_data_compressed['forces'][:3])
    print('forces sad nc', npy_data['forces_SAD'][:3])
    print('forces sad c', npy_data_compressed['forces_SAD'][:3])
    np.save(npy_path, npy_data, allow_pickle=True)

100
results len 0
calc 0
converged SCF energy = -76.3400734696848
calc 1
converged SCF energy = -76.357571836433
calc 2
converged SCF energy = -76.3216300513999
calc 3
converged SCF energy = -76.3301241150047
calc 4
converged SCF energy = -76.3575156401553
calc 5
converged SCF energy = -76.3489200440544
calc 6
converged SCF energy = -76.354418770028
calc 7
converged SCF energy = -76.3382051551203
calc 8
converged SCF energy = -76.3306062649903
calc 9
converged SCF energy = -76.3116804906144
calc 10
converged SCF energy = -76.3540169794951
calc 11
converged SCF energy = -76.321613673482
calc 12
converged SCF energy = -76.3132378254684
calc 13
converged SCF energy = -76.3268942587044
calc 14
converged SCF energy = -76.3591715427401
calc 15
converged SCF energy = -76.3401323466418
calc 16
converged SCF energy = -76.3364041493276
calc 17
converged SCF energy = -76.3512802050817
calc 18
converged SCF energy = -76.3366110706739
calc 19
converged SCF energy = -76.3168892440311
calc 20
converg

calc 62
converged SCF energy = -76.3421387008564
calc 63
converged SCF energy = -76.3444510939799
calc 64
converged SCF energy = -76.344240150403
calc 65
converged SCF energy = -76.3407385731545
calc 66
converged SCF energy = -76.3455792142957
calc 67
converged SCF energy = -76.3432698697042
calc 68
converged SCF energy = -76.3444751329801
calc 69
converged SCF energy = -76.3263131275186
calc 70
converged SCF energy = -76.3451856179845
calc 71
converged SCF energy = -76.3538705499209
calc 72
converged SCF energy = -76.3340275073657
calc 73
converged SCF energy = -76.3401195988275
calc 74
converged SCF energy = -76.3508744498065
calc 75
converged SCF energy = -76.3197035445648
calc 76
converged SCF energy = -76.3539526243354
calc 77
converged SCF energy = -76.3092209274349
calc 78
converged SCF energy = -76.3451200214197
calc 79
converged SCF energy = -76.3376486915115
calc 80
converged SCF energy = -76.3143214091739
calc 81
converged SCF energy = -76.3149005704183
calc 82
converged SCF

In [9]:
np.save(save_path, results, allow_pickle=True)

In [23]:
set_types = ['train', 'valid', 'test']
for set_type in set_types:
    data = np.load('datasets/h2o_small_' + set_type + '_augccpvdz_df_augccpvqzjkfit.npy', allow_pickle=True)
    basis = 'augccpvdz'
    auxbasis = 'augccpvqzjkfit'

    print(len(data))
    save_path = 'datasets/h2o_small_' + set_type + '_dft_augccpvdz_df_hm_dm_oe_calc.npy'
    npy_path = 'datasets/h2o_small_' + set_type + '_dft_augccpvdz_df_hm_dm_oe.npy'
    if os.path.exists(save_path):
        results = list(np.load(save_path, allow_pickle=True))
    else:
        results = []
    print('results len', len(results))
    for i in range(len(results), len(data)):
        print('calc', i)
        start = time.time()
        atom = data[i][0]["atom"] 
        mol = gto.M(atom=atom, basis=basis)
        res = []
        res.append(mol.pack())
        mf = dft.RKS(mol)
        mf.chkfile=False
        mf.xc = 'pbe'
        mf.kernel()
        g = mf.nuc_grad_method()
        gradients = g.grad()
        print('elapsed', time.time() - start)
        #print(mfs[i].mo_coeff)
        res = []
        res.append(mol.pack())
        calc_dict = {}
        calc_dict['mo_coeff'] = mf.mo_coeff
        print('mo_coeff shape', calc_dict['mo_coeff'].shape)
        calc_dict['mo_occ'] = mf.mo_occ
        calc_dict['energy'] = mf.e_tot
        calc_dict['forces'] = -gradients/ase.units.Bohr

        dm1 = mf.make_rdm1(mf.mo_coeff, mf.mo_occ)
        auxmol = df.addons.make_auxmol(mol, auxbasis)

        ints_3c2e = df.incore.aux_e2(mol, auxmol, intor='int3c2e')
        ints_2c2e = auxmol.intor('int2c2e')
        print('ints3c2e shape', ints_3c2e.shape)
        print('ints2c2e shape', ints_2c2e.shape)

        nao = mol.nao
        naux = auxmol.nao
        df_coef = scipy.linalg.solve(ints_2c2e, ints_3c2e.reshape(nao*nao, naux).T)
        df_coef = df_coef.reshape(naux, nao, nao)
        if dm1.ndim > 2:
            df_basis = []
            for j in range(dm1.shape[0]):
                df_basis.append(lib.einsum('Pij,ij->P', df_coef, dm1[j]))
            df_basis = np.stack(df_basis, axis=0)
            print(df_basis.shape)

        else:
            df_basis = lib.einsum('Pij,ij->P', df_coef, dm1)

        calc_dict['df_coeff'] = df_basis
        calc_dict['auxbasis'] = auxbasis
        # print('calc_dict', calc_dict)
        oe = mf.mo_energy
        hm = hf.get_fock(mf)
        calc_dict.update({"mo_energies":oe, "density_matrix": dm1,
                          "hamiltonian_matrix": hm})
        res.append(calc_dict)
        results.append(res)
        if i%10 == 0:
            np.save(save_path, results, allow_pickle=True)
    np.save(save_path, results, allow_pickle=True)
    npy_data = utils.calc_dict_to_npy(results, convert_forces=False, compress_atoms=False)
    npy_data_compressed = utils.calc_dict_to_npy(results, convert_forces=False, compress_atoms=True)
    # print('atom_number nc', npy_data['atom_numbers'][:3])
    # print('atom_number c', npy_data_compressed['atom_numbers'][:3])
    # print('pos nc', npy_data['positions'][:3])
    # print('pos c', npy_data_compressed['positions'][:3])
    # print('forces nc', npy_data['forces'][:3])
    # print('forces c', npy_data_compressed['forces'][:3])
    # print('forces sad nc', npy_data['forces_SAD'][:3])
    # print('forces sad c', npy_data_compressed['forces_SAD'][:3])
    np.save(npy_path, npy_data, allow_pickle=True)

100
results len 0
calc 0
converged SCF energy = -76.3400734696845
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0980784913     0.0409557016     0.0121017383
1 H     0.0672606599    -0.0066006958    -0.0415839145
2 H     0.0308103506    -0.0343553578     0.0294912121
----------------------------------------------
elapsed 1.0583221912384033
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 1
converged SCF energy = -76.3575718364332
--------------- RKS gradients ---------------
         x                y                z
0 O     0.0214100950    -0.0251963467    -0.0228096731
1 H    -0.0177171393     0.0089279046     0.0013439883
2 H    -0.0036976252     0.0162699073     0.0214678347
----------------------------------------------
elapsed 0.9043209552764893
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 2
converged SCF energy = -76.3216300513999
--------------- RKS gradients ---------------
         x            

converged SCF energy = -76.316889244031
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0150419818    -0.0154591291    -0.0594212570
1 H     0.0445570659    -0.0586789736     0.0743301757
2 H    -0.0295156033     0.0741388984    -0.0149114124
----------------------------------------------
elapsed 0.9636361598968506
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 20
converged SCF energy = -76.3487427550272
--------------- RKS gradients ---------------
         x                y                z
0 O     0.0292641709    -0.0269401694    -0.0611003432
1 H    -0.0025063115    -0.0029711870     0.0267349353
2 H    -0.0267599273     0.0299158831     0.0343680282
----------------------------------------------
elapsed 0.882279634475708
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 21
converged SCF energy = -76.3434816617858
--------------- RKS gradients ---------------
         x                y                z
0 

converged SCF energy = -76.3513631265225
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0075408361     0.0615965759     0.0452134020
1 H    -0.0149876557    -0.0360877334    -0.0306092200
2 H     0.0225243962    -0.0255065974    -0.0146046036
----------------------------------------------
elapsed 0.8992900848388672
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 39
converged SCF energy = -76.3492561534529
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0056888167     0.0576026229    -0.0201569581
1 H     0.0210807868    -0.0218605356     0.0052582196
2 H    -0.0153950949    -0.0357398568     0.0149027169
----------------------------------------------
elapsed 0.9142954349517822
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 40
converged SCF energy = -76.3550878390374
--------------- RKS gradients ---------------
         x                y                z


converged SCF energy = -76.341601579584
--------------- RKS gradients ---------------
         x                y                z
0 O     0.1063779849    -0.0256566789     0.0058135750
1 H    -0.0157138314     0.0378531075    -0.0111858820
2 H    -0.0906641680    -0.0121941179     0.0053697003
----------------------------------------------
elapsed 0.9127635955810547
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 58
converged SCF energy = -76.346827618781
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0597569695    -0.0081670227     0.0708810186
1 H     0.0271543807     0.0210237089    -0.0314245803
2 H     0.0326081480    -0.0128587805    -0.0394556228
----------------------------------------------
elapsed 0.8942372798919678
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 59
converged SCF energy = -76.3207576833713
--------------- RKS gradients ---------------
         x                y                z
0 

converged SCF energy = -76.2852743791027
--------------- RKS gradients ---------------
         x                y                z
0 O     0.1469947311     0.1306260900    -0.3860434788
1 H    -0.1501759541    -0.1235986762     0.3627398922
2 H     0.0031768610    -0.0070278261     0.0233058852
----------------------------------------------
elapsed 0.9038975238800049
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 77
converged SCF energy = -76.3497489660285
--------------- RKS gradients ---------------
         x                y                z
0 O     0.0130324242    -0.0198065513    -0.0062733220
1 H     0.0040556597    -0.0217442927     0.0238453745
2 H    -0.0170867306     0.0415487884    -0.0175700694
----------------------------------------------
elapsed 0.9324026107788086
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 78
converged SCF energy = -76.305042974011
--------------- RKS gradients ---------------
         x                y                z
0

...

converged SCF energy = -76.3578698601906
--------------- RKS gradients ---------------
         x                y                z
0 O     0.0044996743     0.0030032756    -0.0230837815
1 H    -0.0039271663    -0.0080722366    -0.0107583106
2 H    -0.0005713603     0.0050695689     0.0338404987
----------------------------------------------
elapsed 0.8607587814331055
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 9
converged SCF energy = -76.3465506552557
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0550590844    -0.0624587310     0.0227952038
1 H     0.0476491681     0.0586644615    -0.0216205838
2 H     0.0074089465     0.0037934378    -0.0011713677
----------------------------------------------
elapsed 0.8967881202697754
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 10
converged SCF energy = -76.3414848922713
--------------- RKS gradients ---------------
         x                y                z
0

converged SCF energy = -76.3492799479839
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0596419050    -0.0778685072    -0.0007971862
1 H     0.0326807177     0.0424457412     0.0677551493
2 H     0.0269591776     0.0354235503    -0.0669581534
----------------------------------------------
elapsed 0.8641579151153564
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 28
converged SCF energy = -76.346473401454
--------------- RKS gradients ---------------
         x                y                z
0 O     0.0469627574     0.0634690720    -0.0394618461
1 H    -0.0331880068    -0.0482798223     0.0286776513
2 H    -0.0137763616    -0.0151897145     0.0107800200
----------------------------------------------
elapsed 0.8796305656433105
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 29
converged SCF energy = -76.3425574742577
--------------- RKS gradients ---------------
         x                y                z
0

converged SCF energy = -76.3340317887761
--------------- RKS gradients ---------------
         x                y                z
0 O     0.0036133385     0.0041645375     0.0010902548
1 H     0.0071365418     0.0659221823     0.0056880981
2 H    -0.0107509565    -0.0700830005    -0.0067748489
----------------------------------------------
elapsed 0.8872606754302979
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 47
converged SCF energy = -76.3190741948432
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0416853465    -0.1197196995     0.0261920260
1 H     0.0324149767     0.1420030517     0.0400499017
2 H     0.0092704931    -0.0222873699    -0.0662444013
----------------------------------------------
elapsed 0.8617141246795654
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 48
converged SCF energy = -76.3567832771493
--------------- RKS gradients ---------------
         x                y                z


converged SCF energy = -76.3328522936395
--------------- RKS gradients ---------------
         x                y                z
0 O     0.0446328457    -0.1326653072    -0.0205508163
1 H     0.0286351049    -0.0320103170     0.0098965752
2 H    -0.0732684220     0.1646813500     0.0106528044
----------------------------------------------
elapsed 0.9031639099121094
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 66
converged SCF energy = -76.3439610660302
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.1118030541     0.0507997555     0.0860118225
1 H     0.0781312887     0.0085304256    -0.0826868810
2 H     0.0336698832    -0.0593334992    -0.0033203569
----------------------------------------------
elapsed 0.8659629821777344
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 67
converged SCF energy = -76.3423473366927
--------------- RKS gradients ---------------
         x                y                z


converged SCF energy = -76.3440883414503
--------------- RKS gradients ---------------
         x                y                z
0 O     0.0671366885     0.0529737632     0.0173271647
1 H    -0.0470915847    -0.0130955611    -0.0559578142
2 H    -0.0200485990    -0.0398741490     0.0386303946
----------------------------------------------
elapsed 0.9336426258087158
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 85
converged SCF energy = -76.3481271918055
--------------- RKS gradients ---------------
         x                y                z
0 O     0.0234923786    -0.0780052639     0.1027879802
1 H    -0.0012258444     0.0112227339    -0.0093373029
2 H    -0.0222670994     0.0667820378    -0.0934498183
----------------------------------------------
elapsed 0.87245774269104
ints3c2e shape (41, 41, 294)
ints2c2e shape (294, 294)
calc 86
converged SCF energy = -76.3396790706261
--------------- RKS gradients ---------------
         x                y                z
0 

In [21]:
set_types = ['train', 'valid', 'test']
for set_type in set_types:
    data1 = np.load('datasets/h2o_small_' + set_type + '_augccpvdz_df_augccpvqzjkfit.npy', allow_pickle=True)
    data2 = np.load('datasets/h2o_small_' + set_type + '_dft_augccpvdz_hm_dm_oe_calc.npy', allow_pickle=True)
    save_path = 'datasets/h2o_small_' + set_type + '_dft_augccpvdz_df_hm_dm_oe_calc.npy'
    for i in range(len(data1)):
        data2[i][1]['auxbasis'] = data1[i][1]['auxbasis']
        data2[i][1]['df_coeff'] = data1[i][1]['df_coeff']

    np.save(save_path, data2, allow_pickle=True)

In [17]:
results = np.load(save_path, allow_pickle=True)
for res in results:
    calc = res[1]
    mo_coeff = calc['mo_coeff']
    mo_en = calc['mo_energies']
    mol = gto.M(atom=res[0]['atom'], basis=basis)
    s1e = mol.intor('int1e_ovlp')
    ks = calc['hamiltonian_matrix'] 
    mf = dft.RKS(mol)
    moe_calc, mo_calc = mf.eig(ks, s1e)
    print('moe calc', moe_calc)
    print('mo en', mo_en)
    print('mo en error kcal', utils.hartree_to_kcal(np.mean(np.abs(moe_calc - mo_en))))
    print('mo en error hartree', np.mean(np.abs(moe_calc - mo_en)))

moe calc [-18.77941342  -0.94186483  -0.4782721   -0.35201599  -0.26678736
  -0.0359311    0.01966254   0.08871976   0.10821145   0.11597369
   0.14045355   0.16647297   0.21631347   0.26261142   0.26726919
   0.2984314    0.42980765   0.46432998   0.51202908   0.58817288
   0.72088599   0.86827459   0.88347438   0.91233964   1.00137169
   1.13292667   1.1551329    1.28143649   1.68199929   1.68887035
   1.81578292   1.99889631   2.03790594   2.26650384   2.36135552
   2.61382456   3.19210097   3.21456527   3.22549721   3.48752511
   3.83322557]
mo en [-18.77941873  -0.94186692  -0.47827374  -0.35201808  -0.2667895
  -0.03593132   0.01966234   0.08871947   0.10821105   0.11597344
   0.14045302   0.16647275   0.21631326   0.26261135   0.26726886
   0.29843109   0.42980748   0.46432973   0.51202841   0.5881725
   0.72088543   0.86827366   0.88347357   0.9123389    1.0013707
   1.13292509   1.15513185   1.28143522   1.68199911   1.68886989
   1.81578233   1.9988957    2.03790525   2.26650

moe calc [-1.87884821e+01 -9.35279070e-01 -4.46016977e-01 -3.71889217e-01
 -2.66379363e-01 -4.10718448e-02  1.83991453e-02  8.50149129e-02
  1.09346647e-01  1.16190937e-01  1.27068273e-01  1.64391582e-01
  2.27533179e-01  2.44588958e-01  2.66964642e-01  2.78998923e-01
  4.34523173e-01  4.56380051e-01  4.96063290e-01  5.67829414e-01
  7.29234154e-01  8.78250166e-01  8.83868350e-01  9.09687820e-01
  9.82587626e-01  1.10776189e+00  1.13372147e+00  1.27405286e+00
  1.65708455e+00  1.67815134e+00  1.80683039e+00  1.82924391e+00
  2.05230547e+00  2.19891537e+00  2.30471471e+00  2.67402328e+00
  3.19477162e+00  3.19878994e+00  3.21137592e+00  3.47201542e+00
  3.73703793e+00]
mo en [-1.87884891e+01 -9.35281663e-01 -4.46019078e-01 -3.71891947e-01
 -2.66382190e-01 -4.10721221e-02  1.83989019e-02  8.50145380e-02
  1.09346108e-01  1.16190528e-01  1.27067682e-01  1.64391256e-01
  2.27533061e-01  2.44588509e-01  2.66964619e-01  2.78998626e-01
  4.34523036e-01  4.56379682e-01  4.96062569e-01  5.67829

In [ ]:
results = []
for i in range(len(mfs)):
    #print(mfs[i].mo_coeff)
    mol_dict = mols[i].pack()
    calc_dict = {}
    calc_dict['mo_coeff'] = mfs[i].mo_coeff
    calc_dict['mo_occ'] = mfs[i].mo_occ
    calc_dict['energy'] = mfs[i].e_tot
    calc_dict['forces'] = forces[i]
    results.append((mol_dict, calc_dict))
    results.append(res)

np.save('datasets/h2o_dynamic_pyscf_631gss_dft_f.npy', results)

In [ ]:
data_scf = np.load('datasets/h2o_dynamic_pyscf_631gss_dft_f.npy', allow_pickle=True)
print(data_scf)
print(len(data_scf))

In [ ]:
data_scf = np.load('datasets/h2o_dynamic_pyscf_dft_f.npy', allow_pickle=True)
print(data_scf)
print(len(data_scf))
for d in data_scf:
    print('energy', d['energy'])
    print('forces', d['forces'])

In [ ]:
for d in data_scf:
    d['forces'] = d['forces'] * 0.529177


np.save('datasets/h2o_dynamic_pyscf_dft_f.npy', data_scf)

In [ ]:
new_data = []
data_scf = data_scf = np.load('datasets/h2o_dynamic_pyscf_dft_f.npy', allow_pickle=True)
for d in data_scf:
    new_d = []
    mo_coeff = d.pop('mo_coeff')
    mo_occ = d.pop('mo_occ')
    en = d.pop('energy')
    f = d.pop('forces')
    new_d.append(d)
    new_d.append({'mo_coeff': mo_coeff, 'mo_occ': mo_occ, 'energy': en, 'forces': f})
    new_data.append(new_d)

np.save('datasets/h2o_dynamic_pyscf_dft_f_en.npy', new_data)

In [ ]:
results = {'E': [], 'F': [], 'R': [], 'z': np.array([8, 1, 1])}
for i in range(len(mfs)):
    #print(mfs[i].mo_coeff)
    res = mols[i].pack()
    pos = []
    for a in res['atom']:
        pos.append(a[1])
    pos = np.array(pos)
    print('pos.shape', pos.shape)
    results['R'].append(pos)
    results['E'].append(mfs[i].e_tot)
    results['F'].append(forces[i])
    
results['R'] = np.array(results['R'])
results['E'] = np.array(results['E'])
results['F'] = np.array(results['F'])

np.savez('datasets/water_pyscf_dft_f', **results)

In [ ]:
results = {'energy': [], 'forces': [], 'positions': [],
           'atom_numbers': [8, 1, 1], 'atom_types': ['O', 'H', 'H'],
          'mo_coeff': [], 'mo_occ': []}
for d in data_scf:
    #print(mfs[i].mo_coeff)
    pos = []
    for a in d['atom']:
        pos.append(a[1])
    pos = np.array(pos)
    print('pos.shape', pos.shape)
    results['positions'].append(pos)
    results['energies'].append(d['energy'])
    results['forces'].append(d['forces'])
    results['mo_coeff'].append(d['mo_coeff'])
    results['mo_occ'].append(d['mo_occ'])

for key in results.keys():
    results[key] = np.array(results[key])
    
np.savez('datasets/h2o_dynamic_pyscf_dft_f', **results)

In [ ]:
print(np.load('datasets/h2o_dynamic_pyscf_dft.npy', allow_pickle=True)[0])
print(np.load('datasets/h2o_dynamic_pyscf_dft_f.npy', allow_pickle=True)[0])
print(np.load('datasets/h2o_dynamic_pyscf_dft_f_en.npy', allow_pickle=True)[0])